# Text Encoding Repair Workflow

This notebook targets legacy text files like the sample `Музыка.txt`. It does two things:

1. Detect whether the file looks broken or simply uses a non-UTF-8 encoding.
2. Repair the file by decoding it with the right source encoding and writing UTF-8 output.

For your sample file, the bytes look consistent with `cp1251`, so the likely fix is conversion to UTF-8 rather than reconstructing random corruption.


In [31]:
from pathlib import Path
import pandas as pd

from encoding_tools import analyze_file, detect_encoding_issue, fix_file_encoding

TARGET_FILE = Path('/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/ban.txt')
EXPECTED_ENCODING = 'cp1251'
SCRIPT_HINT = 'cyrillic'
TARGET_ENCODING = 'utf-8'
OUTPUT_FILE = TARGET_FILE.with_name(f'{TARGET_FILE.stem}.utf8{TARGET_FILE.suffix}')

TARGET_FILE.exists(), OUTPUT_FILE


(True,
 PosixPath('/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/ban.utf8.txt'))

In [32]:
candidate_results = analyze_file(
    TARGET_FILE,
    script_hint=SCRIPT_HINT,
)

pd.DataFrame(candidate_results)


,encoding,can_decode,suspicious_score,script_ratio,replacement_count,control_count,mojibake_count,sample,error
0,cp1251,True,0.02,0.998,0,0,0,просмотр видосиков на ютубчике без запоминания...,NaN
1,koi8-r,True,0.02,0.998,0,0,0,ОПНЯЛНРП БХДНЯХЙНБ МЮ ЧРСАВХЙЕ АЕГ ГЮОНЛХМЮМХЪ...,NaN
2,mac_cyrillic,True,0.02,0.998,0,0,0,просмотр видосиков на ютубчике без запоминани€...,NaN
3,cp1252,True,91.00,0.000,0,0,7,ïðîñìîòð âèäîñèêîâ íà þòóá÷èêå áåç çàïîìèíàíèÿ...,NaN
4,latin-1,True,91.00,0.000,0,0,7,ïðîñìîòð âèäîñèêîâ íà þòóá÷èêå áåç çàïîìèíàíèÿ...,NaN
5,cp866,True,390.02,0.998,0,0,39,яЁюёьюЄЁ тшфюёшъют эр ■Єєсўшъх схч чряюьшэрэш ...,NaN
6,utf-8,False,inf,0.000,0,0,0,,'utf-8' codec can't decode byte 0xef in positi...
7,utf-8-sig,False,inf,0.000,0,0,0,,'utf-8' codec can't decode byte 0xef in positi...


In [33]:
detection = detect_encoding_issue(
    TARGET_FILE,
    expected_encoding=EXPECTED_ENCODING,
    script_hint=SCRIPT_HINT,
)

detection


{'path': '/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/ban.txt',
 'script_hint': 'cyrillic',
 'candidate_results': [{'encoding': 'cp1251',
   'can_decode': True,
   'suspicious_score': 0.02,
   'script_ratio': 0.998,
   'replacement_count': 0,
   'control_count': 0,
   'mojibake_count': 0,
   'sample': 'просмотр видосиков на ютубчике без запоминания информации  мысленные сладости   "мысленные',
   'error': None},
  {'encoding': 'koi8-r',
   'can_decode': True,
   'suspicious_score': 0.02,
   'script_ratio': 0.998,
   'replacement_count': 0,
   'control_count': 0,
   'mojibake_count': 0,
   'sample': 'ОПНЯЛНРП БХДНЯХЙНБ МЮ ЧРСАВХЙЕ АЕГ ГЮОНЛХМЮМХЪ ХМТНПЛЮЖХХ  ЛШЯКЕММШЕ ЯКЮДНЯРХ   "ЛШЯКЕММШЕ',
   'error': None},
  {'encoding': 'mac_cyrillic',
   'can_decode': True,
   'suspicious_score': 0.02,
   'script_ratio': 0.998,
   'replacement_count': 0,
   'control_count': 0,
   'mojibake_count': 0,
   'sample': 'просмотр видосиков на ютубчике без запоминани€ информации  мы

In [34]:
import importlib
import encoding_tools

importlib.reload(encoding_tools)
fix_file_encoding = encoding_tools.fix_file_encoding

In [35]:
result = fix_file_encoding(
    TARGET_FILE,
    source_encoding=EXPECTED_ENCODING,
    target_encoding=TARGET_ENCODING,
    output_path=OUTPUT_FILE,
    overwrite=False,
    preserve_metadata=True,
    recover_mojibake=False,
    script_hint=SCRIPT_HINT,
)

result

{'input_path': '/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/ban.txt',
 'output_path': '/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/ban.utf8.txt',
 'backup_path': None,
 'source_encoding': 'cp1251',
 'target_encoding': 'utf-8',
 'preserve_metadata': True,
 'metadata_preserved': True,
 'metadata_error': None,
 'recover_mojibake': False,
 'recovery_changed_text': False,
 'recovery_strategy': 'not_applied',
 'analysis': {'path': '/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/ban.txt',
  'script_hint': 'cyrillic',
  'candidate_results': [{'encoding': 'cp1251',
    'can_decode': True,
    'suspicious_score': 0.02,
    'script_ratio': 0.998,
    'replacement_count': 0,
    'control_count': 0,
    'mojibake_count': 0,
    'sample': 'просмотр видосиков на ютубчике без запоминания информации  мысленные сладости   "мысленные',
    'error': None},
   {'encoding': 'koi8-r',
    'can_decode': True,
    'suspicious_score': 0.02,
    '

In [36]:
fixed_preview = OUTPUT_FILE.read_text(encoding=TARGET_ENCODING).splitlines()[:12]
fixed_preview


['просмотр видосиков на ютубчике без запоминания информации',
 'мысленные сладости',
 '\t"мысленные сладости" - это неправильно интерпретированные моей психикой процессы происходящие внутри меня',
 '\t- это неправильная реакция на изменения состояния внутри меня',
 '\t- искаженая реакция(неверная образ/ассоциация) на происходящие внешние процессы/события',
 '\t- это реакция(приводящая к отрицательному результату), построенная многократным повторением на различне события',
 'когда сталкиваюсь со сложной темы в обучении и если сразу ее не освою, то забрасываю обучение на несколько дней',
 'уход мысли в другую сторону во время запоминания/обучения',
 'когда долго не могу подобрать образ под информацию - мысль уходит',
 'трогать лицо когда делаю что-нибудь(читаю, думаю, запоминаю)',
 '\tиногда делаю это чтобы занять чем то руки когда ничего в руках не держу',
 '\tкогда задумаюсь над чем-либо']

## CLI Scripts

The workspace also contains two standalone scripts:

- `detect_broken_encoding.py` checks candidate encodings and reports whether the file is likely broken.
- `fix_text_encoding.py` converts the file to UTF-8 and can optionally attempt mojibake recovery.
- Output metadata is preserved by default (mtime/atime/mode when the platform allows it).

Example commands:

```bash
python detect_broken_encoding.py '/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/Музыка.txt' --expected-encoding cp1251
python fix_text_encoding.py '/Users/kirillmaier/DocLocal/bak_1tb/back win 7/Documents/Documents/Музыка.txt' --source-encoding cp1251
```